# Retrain Stage 1 -- clean before/after-Task-2 comparison, with ROI selection fixes

Runs the `meeting-prep` branch. Compared to the Task 2 retrain (`Av2_retrain_task2.ipynb`), this fixes the
things that made that run's numbers hard to trust, then trains **both** feature conventions under identical
conditions so the Task 2 comparison is actually fair:

| | before | now |
|---|---|---|
| agents fed to the model | valid in *every* history frame, any distance, first 32 in array order | valid **now**, within **50 m**, **nearest 32**; dropouts in the 1 s history interpolated + flagged (7th feature) |
| map points | first **200** within 80 m in *array* order (arbitrary subset) | **all** within 80 m, nearest-first (cap 6000, measured below) |
| objects loaded | `max_num_objects=32` -- first 32 in file order, SDC not guaranteed | `128` (all of WOMD) + SDC presence checked |
| LR schedule | `T_max=5` but trained 10 epochs -> LR rose again for epochs 6-9 | `T_max = NUM_EPOCHS` |
| test metrics | computed on the *last* epoch's model | computed on the **saved best** checkpoint |
| Task 2 comparison | two different runs (5 vs 10 epochs, different schedules) | **one loop, same batches, same init**, only the feature convention differs |

**Runtime**: both variants share data loading, but extraction runs twice and the map branch now sees far more
points. Expect roughly **1.5-2.5 h on a T4** for 10 epochs. Lower `NUM_EPOCHS` / `TRAIN_SCENARIOS` below if
you need it faster -- just keep them identical for both variants (they are, by construction).

Outputs: `weights/stage1_roi_tau.pth`, `weights/stage1_roi_t.pth`, `results/retrain_meeting_prep.json`, and a
markdown table printed at the end for the README.

## Setup

In [ ]:
from google.colab import auth
auth.authenticate_user()
!git config --global user.email "mayankarya510@gmail.com"
!git config --global user.name "meyenk"

In [ ]:
!pip install -q git+https://github.com/waymo-research/waymax.git@main#egg=waymo-waymax

Clones the `meeting-prep` branch, not `main` -- the changes this notebook tests aren't merged yet.

In [ ]:
!git clone -b meeting-prep https://github.com/meyenk/waymax_experiment.git
%cd waymax_experiment
!git log --oneline -3

**(Optional) push access** -- only needed to push results/weights back from here. Skip if you'd rather download and commit yourself.

In [ ]:
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/meyenk/waymax_experiment.git

## Sanity check (synthetic tests, no Waymax needed)
If any of these fail, stop -- the checkout is wrong or something regressed.

In [ ]:
!python -m tests.test_features
!python -m tests.test_transforms
!python -m tests.test_mpc
!python -m tests.test_closed_loop

## Data + diagnostics (before spending GPU time)

In [ ]:
import dataclasses, json, os, time
import numpy as np
import torch
import jax
from waymax import dataloader, config as _config

from src import features as F
from src.dataset import val_test_split
jax.config.update("jax_enable_x64", True)

# max_num_objects=128 = every object WOMD has. The old notebooks used 32, and
# Waymax keeps the first 32 in FILE order (not nearest, and with no guarantee
# the SDC is among them -- see the check below).
MAX_NUM_OBJECTS = 128
train_config = dataclasses.replace(
    _config.WOD_1_1_0_TRAINING,
    path="gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/training/training_tfexample.tfrecord@1000",
    max_num_objects=MAX_NUM_OBJECTS,
)
val_config = dataclasses.replace(
    _config.WOD_1_1_0_VALIDATION,
    path="gs://waymo_open_dataset_motion_v_1_1_0/uncompressed/tf_example/validation/validation_tfexample.tfrecord@150",
    max_num_objects=MAX_NUM_OBJECTS,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"using device: {device}")
print(f"ROI: agents within {F.AGENT_RADIUS} m (max {F.MAX_AGENTS}), "
      f"map within {F.MAP_RADIUS} m (max {F.MAX_MAP_POINTS} points), lights max {F.MAX_LIGHTS}")

print("loading validation scenarios (splitting into val/test, 70/30)...")
val_scenarios, test_scenarios = [], []
for i, scenario in enumerate(dataloader.simulator_state_generator(config=val_config)):
    if len(val_scenarios) + len(test_scenarios) >= 1000:
        break
    (val_scenarios if val_test_split(i) == "val" else test_scenarios).append(scenario)
print(f"  val scenarios: {len(val_scenarios)}, test scenarios: {len(test_scenarios)}")

### Was the SDC ever missing under the old `max_num_objects=32`?
Reloads 300 validation scenarios with the old cap and counts how many no longer contain the SDC. For those,
`np.argmax(is_sdc)` returned 0 -- the old pipeline silently treated some other object as the ego.

In [ ]:
old_cfg = dataclasses.replace(val_config, max_num_objects=32)
n_checked, n_missing = 0, 0
for i, s in enumerate(dataloader.simulator_state_generator(config=old_cfg)):
    if i >= 300:
        break
    n_checked += 1
    n_missing += int(not np.array(s.object_metadata.is_sdc).any())
n_missing_128 = sum(int(not np.array(s.object_metadata.is_sdc).any()) for s in val_scenarios + test_scenarios)
print(f"max_num_objects=32 : SDC missing in {n_missing}/{n_checked} scenarios ({100*n_missing/max(n_checked,1):.1f}%)")
print(f"max_num_objects=128: SDC missing in {n_missing_128}/{len(val_scenarios)+len(test_scenarios)}")
diagnostics = {"sdc_missing_cap32": [n_missing, n_checked], "sdc_missing_cap128": n_missing_128}

### What does the new selection actually change?
Over 150 validation scenarios, per decision instant: agents picked under the new vs old rule, how many of the
new picks the old rule would have **dropped** (a dropout in the window), how many agents the old rule fed the
model from **outside** the ROI, how much of the history is filled, and how many map points fall in range.

**Check `map cap binds`**: if it's more than a few %, raise `F.MAX_MAP_POINTS` here (it isn't part of the
weights, so any value is fine) -- the aim is to cover everything in the radius.

In [ ]:
from src.visualize import selection_report
rep = selection_report(val_scenarios[:150])
pct = lambda a: np.percentile(a, [50, 90, 99]).round(0).astype(int).tolist()
summary = {
    "agents picked (new), mean": float(rep["agents_new"].mean()),
    "agents fed (old rule), mean": float(rep["agents_old"].mean()),
    "new picks the old rule dropped (dropout), mean": float(rep["agents_rescued"].mean()),
    "decision instants with >=1 rescued agent (%)": float(100 * (rep["agents_rescued"] > 0).mean()),
    "old-rule agents outside the ROI, mean": float(rep["agents_far_old"].mean()),
    "history frames filled among picked agents (%)": float(100 * rep["frac_frames_filled"].mean()),
    "map points in radius p50/p90/p99": pct(rep["map_in_radius"]),
    "map cap binds (% of instants)": float(100 * (rep["map_in_radius"] > F.MAX_MAP_POINTS).mean()),
}
for k, v in summary.items():
    print(f"  {k:52s} {v if isinstance(v, list) else round(v, 2)}")
diagnostics["selection"] = summary

### See it
Left: world frame -- coloured map points/agents are exactly what the model gets; red rings = agents the old
rule would have dropped; `x` = filled frames. Right: the agent tensor itself, in the ego frame.

In [ ]:
from src.visualize import plot_selection, pick_dynamic_scenario, pick_turning_scenario
demo = pick_dynamic_scenario(val_scenarios, min_displacement=35.0)
turn = pick_turning_scenario(val_scenarios, min_heading_change_deg=25.0, min_displacement=5.0)
for s in (demo, turn):
    for t in (19, 49):
        plot_selection(s, t)

## Train both variants on identical batches
- `tau` = time-matched features (Task 2, the current convention)
- `t`   = decision-instant anchoring (pre-Task 2), kept only for this comparison

Same seed -> identical initial weights; same scenarios in the same order; separate optimizers and schedulers
with `T_max = NUM_EPOCHS`. Each variant's best-val checkpoint is saved separately.

In [ ]:
from src.model import Stage1Model
from src.train import run_epoch_variants
import src.train as train_mod

NUM_EPOCHS = 10
TRAIN_SCENARIOS = 2000   # same as the earlier runs
BATCH_SIZE = 32
LR = 1e-3
SEED = 0
os.makedirs("weights", exist_ok=True)
os.makedirs("results", exist_ok=True)
CKPT = {"tau": "weights/stage1_roi_tau.pth", "t": "weights/stage1_roi_t.pth"}

variants = {}
for name, tm in (("tau", True), ("t", False)):
    torch.manual_seed(SEED)
    m = Stage1Model(hidden_dim=64, future_len=30).to(device)
    opt = torch.optim.Adam(m.parameters(), lr=LR)
    variants[name] = {"model": m, "optimizer": opt, "time_matched": tm,
                      "scheduler": torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=NUM_EPOCHS)}
assert all(torch.equal(a, b) for a, b in zip(variants["tau"]["model"].parameters(),
                                             variants["t"]["model"].parameters())), "inits should match"

history = {n: {"train": [], "val": [], "lr": []} for n in variants}
best = {n: {"val_loss": float("inf"), "epoch": None} for n in variants}
t_start = time.time()
for epoch in range(NUM_EPOCHS):
    print(f"\n--- epoch {epoch} ({(time.time() - t_start) / 60:.0f} min elapsed) ---")
    tr = run_epoch_variants(variants, dataloader.simulator_state_generator(config=train_config),
                            max_scenarios=TRAIN_SCENARIOS, batch_size=BATCH_SIZE, train=True,
                            device=device, verbose_every=25)
    va = run_epoch_variants(variants, val_scenarios, max_scenarios=len(val_scenarios),
                            batch_size=BATCH_SIZE, train=False, device=device, verbose_every=0)
    for n, v in variants.items():
        history[n]["lr"].append(v["optimizer"].param_groups[0]["lr"])
        v["scheduler"].step()
        history[n]["train"].append(tr[n]); history[n]["val"].append(va[n])
        flag = ""
        if va[n]["loss"] < best[n]["val_loss"]:
            best[n] = {"val_loss": va[n]["loss"], "epoch": epoch}
            torch.save(v["model"].state_dict(), CKPT[n])
            flag = "  -> new best, saved"
        print(f"  {n:4s} train loss={tr[n]['loss']:.4f} ade={tr[n]['ade']:.3f} fde={tr[n]['fde']:.3f} | "
              f"val loss={va[n]['loss']:.4f} ade={va[n]['ade']:.3f} fde={va[n]['fde']:.3f} "
              f"| lr={history[n]['lr'][-1]:.2e}{flag}")
print(f"\nmap cap bound on {100 * tr['tau']['map_truncated_frac']:.1f}% of training examples (last epoch)")
print(f"scenarios skipped for a missing SDC: {train_mod.SKIPPED_NO_SDC}")
print(f"best epochs: {best}")

In [ ]:
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for n, c in (("tau", "C0"), ("t", "C1")):
    for split, ls in (("train", "-"), ("val", "--")):
        axes[0].plot([h["loss"] for h in history[n][split]], ls=ls, c=c, label=f"{n} {split}")
        axes[1].plot([h["fde"] for h in history[n][split]], ls=ls, c=c, label=f"{n} {split}")
    axes[2].plot(history[n]["lr"], c=c, label=n)
for ax, t in zip(axes, ("MSE loss", "FDE (m)", "learning rate (should fall monotonically now)")):
    ax.set_title(t); ax.set_xlabel("epoch"); ax.legend(fontsize=8)
plt.tight_layout(); plt.show()

## Test set -- on the saved best checkpoints
Overall, and split into turning vs straight scenarios (Task 2's fix was expected to matter most on turns).

In [ ]:
from src.visualize import is_turning_scenario
for n, v in variants.items():
    v["model"].load_state_dict(torch.load(CKPT[n], map_location=device))
    v["model"].eval()
turning = [s for s in test_scenarios if is_turning_scenario(s)]
straight = [s for s in test_scenarios if not is_turning_scenario(s)]
print(f"test: {len(test_scenarios)} scenarios ({len(turning)} turning, {len(straight)} straight)")

test_results = {}
for subset_name, subset in (("all", test_scenarios), ("turning", turning), ("straight", straight)):
    r = run_epoch_variants(variants, subset, max_scenarios=len(subset), batch_size=BATCH_SIZE,
                           train=False, device=device, verbose_every=0)
    test_results[subset_name] = r
    for n in variants:
        print(f"  {subset_name:8s} {n:4s} loss={r[n]['loss']:.4f} ADE={r[n]['ade']:.3f} m "
              f"FDE={r[n]['fde']:.3f} m  (n={r[n]['n_examples']})")

## Does the model actually use the scene? (branch ablation, tau model)
Each row hides whole encoder branches at inference (their items are masked out, so the pool returns zeros).
If hiding agents+map+lights barely moves ADE/FDE, the model is mostly extrapolating its own speed -- which
would explain the "keeps doing what it's doing" bias.

In [ ]:
ablations = [(), ("agents",), ("map",), ("lights",), ("agents", "map", "lights")]
tau_only = {"tau": variants["tau"]}
ablation_results = {}
for subset_name, subset in (("all", test_scenarios), ("turning", turning)):
    for ab in ablations:
        r = run_epoch_variants(tau_only, subset, max_scenarios=len(subset), batch_size=BATCH_SIZE,
                               train=False, device=device, verbose_every=0, ablate=ab)["tau"]
        key = "+".join(ab) or "none"
        ablation_results[f"{subset_name}/{key}"] = r
        print(f"  {subset_name:8s} hide {key:22s} ADE={r['ade']:.3f} m  FDE={r['fde']:.3f} m")

## Look at predictions + what the trained model attends to

In [ ]:
from src.visualize import plot_world_frame_snapshots, play_scenario_clip
turn_test = pick_turning_scenario(test_scenarios, min_heading_change_deg=25.0, min_displacement=5.0)
play_scenario_clip(turn_test)
for n in variants:
    print(f"--- {n} ---")
    plot_world_frame_snapshots(variants[n]["model"], turn_test, num_snapshots=6, window=80.0, device=device)
# colours = the model's learned relevance weight per agent / map point
plot_selection(turn_test, 39, model=variants["tau"]["model"], device=device)

## Save results + README table

In [ ]:
results = {
    "config": {"num_epochs": NUM_EPOCHS, "train_scenarios": TRAIN_SCENARIOS, "batch_size": BATCH_SIZE,
               "lr": LR, "seed": SEED, "max_num_objects": MAX_NUM_OBJECTS,
               "agent_radius": F.AGENT_RADIUS, "max_agents": F.MAX_AGENTS, "map_radius": F.MAP_RADIUS,
               "max_map_points": F.MAX_MAP_POINTS},
    "best": best, "history": history, "test": test_results, "ablation": ablation_results,
    "diagnostics": diagnostics,
}
with open("results/retrain_meeting_prep.json", "w") as f:
    json.dump(results, f, indent=1, default=float)

lines = ["| test subset | variant | ADE (m) | FDE (m) | examples |", "|---|---|---|---|---|"]
for sub, r in test_results.items():
    for n in ("t", "tau"):
        lines.append(f"| {sub} | {'pre-Task-2 (t)' if n == 't' else 'Task 2 (tau)'} | "
                     f"{r[n]['ade']:.3f} | {r[n]['fde']:.3f} | {r[n]['n_examples']} |")
lines += ["", "| hidden branches (tau, all test) | ADE (m) | FDE (m) |", "|---|---|---|"]
for k, r in ablation_results.items():
    if k.startswith("all/"):
        lines.append(f"| {k[4:]} | {r['ade']:.3f} | {r['fde']:.3f} |")
print("\n".join(lines))

**(Optional) commit + push the checkpoints and results** -- requires the push-access cell above.

In [ ]:
!git add weights/stage1_roi_tau.pth weights/stage1_roi_t.pth results/retrain_meeting_prep.json
!git commit -m "Retrain on ROI selection: tau vs t under identical conditions"
!git push origin meeting-prep

If you didn't push: download `weights/stage1_roi_tau.pth` (Files panel) -- the closed-loop notebook needs it.